In [ ]:
import pandas as pd
from dask.array import average
from gototile.grid import SkyGrid
from goto_sim.scheduling import HEATSurvey
from goto_sim.utils import load_tilelist
from astropy.time import Time, TimeDelta
import numpy as np
import matplotlib.pyplot as plt
import glob
plt.rcParams['font.family'] = 'Nimbus Sans'

In [ ]:
data_dir = "sims"
figure_dir = "figures"
file_pattern = f"{data_dir}/simulation_results_reserve_*.csv"
file_list = glob.glob(file_pattern)
if not file_list:
    raise FileNotFoundError(f"No files found matching pattern: {file_pattern}")
print(f"Found {len(file_list)} files matching the pattern.")
reserve_fractions = [float(x .split('_')[-1].split('.csv')[0]) for x in file_list]
results = {}
for file_path, reserve_fraction in zip(file_list, reserve_fractions):
    df = pd.read_csv(file_path, comment='#')
    results[reserve_fraction] = df
results_df = pd.concat(results, names=['reserve_fraction', 'index']).reset_index(level='reserve_fraction')

heats_survey = HEATSurvey()
heats_tiles = heats_survey.tiles
cold_survey = heats_survey.generate_colds()
cold_tiles = cold_survey.get_tiles()
grid: SkyGrid = SkyGrid.from_name("GOTO")


In [ ]:
# Replace HEATS target tiles with HEATS, COLDs tiles with COLDs.
# This is fairly slow, probably a faster way to do this with mapping
results_df['target'] = results_df['target'].replace({tile: 'HEATS' for tile in heats_tiles})
results_df['target'] = results_df['target'].replace({tile: 'COLDs' for tile in cold_tiles})
results_df

In [ ]:
results_df['target'].value_counts()

In [ ]:
cadence = results_df.groupby(['tile', 'reserve_fraction'])['time'].apply(
    lambda x: np.median(np.diff(np.sort(x)))
).sort_index()
print(f"Total results: {len(cadence)}")
cold_cadences = cadence.loc[cold_tiles]
print(f"COLD results: {len(cold_cadences)}")

heats_cadences = cadence.loc[heats_tiles]
print(f"HEATS results: {len(heats_cadences)}")


In [ ]:
average_cold_cadence = cold_cadences.groupby('reserve_fraction').mean()
average_heats_cadence = heats_cadences.groupby('reserve_fraction').mean()
plt.figure(figsize=(10, 6))
plt.plot(average_cold_cadence.index/2, average_cold_cadence.values, label='COLD Survey', marker='o', color='tab:blue')
plt.plot(average_heats_cadence.index/2, average_heats_cadence.values, label='HEATS Survey', marker='o', color='tab:red')
plt.xlabel('Fraction of Array Time Reserved for Other Surveys')
plt.ylabel('Median Cadence (Days)')
plt.title('Average Cadence vs Reserve Fraction for COLD and HEATS Surveys')
plt.yticks(minor=True, ticks=np.arange(0, 25, 1))
plt.ylim(0, 25)
plt.legend()
plt.grid()
plt.savefig(f"{figure_dir}/average_cadence_vs_reserve_fraction.png", dpi=300)
plt.show()

In [ ]:
# Print a table of the results for COLD survey
print("COLD Survey Average Cadence:")
for i, row in average_cold_cadence.items():
    print(f"Reserve Fraction (total array): {i/2:.2f}, Average Cadence: {row:05.2f} days, Reserve Fraction (of COLDS): {i:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(average_cold_cadence.index/2, average_cold_cadence.values, label='COLD Survey', marker='o', color='tab:blue')
ax.plot(average_heats_cadence.index/2, average_heats_cadence.values, label='HEATS Survey', marker='o', color='tab:red')
ax.set_xlabel('Array Time Fraction Reserved for Other Surveys')
ax.set_ylabel('Median Cadence (Days)')
ax.set_title('Average Cadence vs Reserve Fraction for COLD and HEATS Surveys')
# Plot fraction of array time for COLDS survey
ax.secondary_xaxis('top', functions=(lambda x: 0.5-x, lambda x: 0.5-x)).set_xlabel('COLDs Array Time')
ax.legend()
plt.yticks(minor=True, ticks=np.arange(0, 25, 1))
plt.ylim(0, 25)
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Bar chart of number of observations per target for each reserve fraction
observations_per_target = results_df.groupby(['reserve_fraction', 'target']).size().unstack(fill_value=0)
observations_per_target = observations_per_target.iloc[:, [1, 3, 2, 0]]
observations_per_target

In [ ]:
fraction_per_target = observations_per_target.div(observations_per_target.sum(axis=1), axis=0)
fraction_per_target
array_fraction_per_target = fraction_per_target.copy()
array_fraction_per_target.index = array_fraction_per_target.index / 2
array_fraction_per_target['colds_cadence'] = average_cold_cadence.values
array_fraction_per_target['heats_cadence'] = average_heats_cadence.values
array_fraction_per_target['reserve_fraction'] = array_fraction_per_target.index
array_fraction_per_target

In [ ]:
ax = array_fraction_per_target.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6),
    color=['tab:red', 'tab:grey', 'tab:green', 'tab:blue'],
    alpha=1
)
ax.set_xlabel('Fraction of Array Time Reserved for Other Surveys')
ax.set_ylabel('Fraction of Observations')
ax.set_title('Fraction of Observations per Target vs Reserve Fraction')
for container in ax.containers:
    ax.bar_label(container, fmt='{:.0%}', label_type='center', fontsize=11, color='white')
plt.legend(title='Target', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f"{figure_dir}/array_fraction_barchart.png", dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(average_cold_cadence.index/2, average_cold_cadence.values, label='COLD Survey', marker='o', color='tab:blue')
ax.plot(average_heats_cadence.index/2, average_heats_cadence.values, label='HEATS Survey', marker='o', color='tab:red')
ax.set_xlabel('Array Time Fraction Reserved for Other Surveys')
ax.set_ylabel('Median Cadence (Days)')
ax.set_title('Average Cadence vs Reserve Fraction for COLD and HEATS Surveys')
# Plot fraction of array time for COLDS survey
ax.secondary_xaxis('top', functions=(lambda x: 0.5-x, lambda x: 0.5-x)).set_xlabel('COLDs Array Time')
ax.legend()
plt.yticks(minor=True, ticks=np.arange(0, 25, 1))
plt.ylim(0, 25)
plt.legend()
plt.grid()

bar_axis = ax.twinx()
bar_width = 0.02
array_fraction_per_target.plot(
    kind='bar',
    stacked=True,
    ax=bar_axis,
    width=bar_width,
    color=['tab:red', 'tab:grey', 'tab:green', 'tab:blue'],
    legend=False,
    alpha=0.3
)
bar_axis.set_ylabel('Fraction of Observations')
bar_axis.set_ylim(0, 1)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x=array_fraction_per_target['reserve_fraction'], y=array_fraction_per_target['colds_cadence'], s=1)